In [ ]:
# - Choose the model whose feature selection you want to debug.
# - Paste an execution ID from your configured training pipeline, or its full ARN.
# - These inputs do nothing until you run the preparation cell.
MODEL_ID = "usecase2"
EXECUTION_ID = ""


# Debug feature selection

Run the setup cell, then the short execution cell. The helper reads the actual
FeatureSelection job and its raw data, including cached producer paths. Selection
uses **your current checkout's code and parameters**, in the notebook kernel.
The setup itself does not run selection.

The execution cell prints a complete **VS Code main block**, even if the model
raises an exception or you interrupt it. Copy that output to the bottom of
`mlops-models/src/mlops_models/entrypoints/feature_selection.py`, replacing the
previous local main block. Open the repository root in VS Code, keep that
entrypoint active, and choose **Python: Debug current model entrypoint** with F5.
The existing `.vscode/launch.json` supplies `PYTHONPATH` and launches the module.

Rerun the execution cell after editing base/model selection code. Rerun setup
after changing the model/execution inputs. Restart the kernel for shared
dependency or checkout changes. Your notebook needs the model's dependencies.
The setup and export code live in [the helper](../source/feature_selection/debug.py).


In [ ]:
# - Resolve the selected job and download/reuse its raw input.
# - Prepare a fresh output folder and a standalone VS Code export.
# - Keep selection unexecuted until the next cell.
step = None
from pathlib import Path
import runpy

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "mlops-models/src/mlops_models").is_dir())
debug = runpy.run_path(str(ROOT / "notebooks/source/feature_selection/debug.py"))
step = debug["prepare"](ROOT, MODEL_ID, EXECUTION_ID)


In [ ]:
# - Run the current local selection entrypoint and preserve its returned value.
# - Keep the original traceback if selection fails; do not suppress the error.
# - Always print the prepared VS Code block, including after interruption.
result = None
if step is None:
    raise ValueError("Run the preparation cell successfully first.")
try:
    with debug["execution"](step) as execute:
        result = execute(
            model_id=step.model_id,
            input_path=step.data_directory,
            output_dir=step.output_directory,
        )
    print("Returned value:", result)
finally:
    print(step.vscode_code, flush=True)
